# HR Analytics — Exploratory Data Analysis
**Dataset:** IBM HR Analytics Employee Attrition & Performance  
**Author:** Saketh Reddy  
**Goal:** Understand the key drivers of employee attrition through statistical exploration and visualisation.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

## 2. Load Data

In [ ]:
df = pd.read_csv('data/HR_Employee_Attrition.csv')
print(f'Shape: {df.shape}')
df.head()

## 3. Dataset Overview

In [ ]:
df.info()

In [ ]:
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std', 'max'])

## 4. Data Quality Check

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values found.')

print(f'\nDuplicate rows: {df.duplicated().sum()}')

# Columns with a single constant value (no variance — drop candidates)
constant_cols = [c for c in df.columns if df[c].nunique() == 1]
print(f'Constant columns (no variance): {constant_cols}')
if constant_cols:
    df.drop(columns=constant_cols, inplace=True)
    print(f'Dropped: {constant_cols}')

In [ ]:
# Encode target for correlation analysis
df['Attrition_flag'] = (df['Attrition'] == 'Yes').astype(int)

num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Numerical columns ({len(num_cols)}): {num_cols}')
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')

## 5. Target Variable — Attrition Distribution

In [ ]:
attrition_counts = df['Attrition'].value_counts()
attrition_pct = df['Attrition'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(attrition_counts.index, attrition_counts.values,
            color=['#2196F3', '#F44336'], edgecolor='white', width=0.5)
axes[0].set_title('Attrition Count')
axes[0].set_ylabel('Number of Employees')
for i, v in enumerate(attrition_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

axes[1].pie(attrition_counts.values, labels=attrition_counts.index,
            autopct='%1.1f%%', colors=['#2196F3', '#F44336'],
            startangle=90, wedgeprops=dict(edgecolor='white'))
axes[1].set_title('Attrition Rate')

plt.suptitle('Employee Attrition Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Attrition Rate: {attrition_pct["Yes"]:.2f}%  |  '
      f'Retained: {attrition_pct["No"]:.2f}%')

## 6. Univariate Analysis

In [ ]:
# Distributions of key numerical features
key_nums = ['Age', 'MonthlyIncome', 'YearsAtCompany', 'TotalWorkingYears',
            'DistanceFromHome', 'YearsSinceLastPromotion']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(key_nums):
    axes[i].hist(df[col], bins=25, color='#2196F3', edgecolor='white', alpha=0.85)
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.2,
                    label=f'Mean: {df[col].mean():.1f}')
    axes[i].axvline(df[col].median(), color='orange', linestyle='--', linewidth=1.2,
                    label=f'Median: {df[col].median():.1f}')
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

plt.suptitle('Distribution of Key Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Categorical feature distributions
key_cats = ['Department', 'JobRole', 'Gender', 'MaritalStatus', 'EducationField', 'BusinessTravel']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(key_cats):
    counts = df[col].value_counts()
    axes[i].barh(counts.index, counts.values, color='#42A5F5', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('Count')
    for j, v in enumerate(counts.values):
        axes[i].text(v + 5, j, str(v), va='center', fontsize=8)

plt.suptitle('Distribution of Key Categorical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Bivariate Analysis — Attrition vs Key Features

In [ ]:
def attrition_rate_by(col, title=None, figsize=(8, 4)):
    """Plot attrition rate (%) for each category in col."""
    rates = (df.groupby(col)['Attrition_flag'].mean() * 100).sort_values(ascending=False)
    counts = df.groupby(col)['Attrition_flag'].count()

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar(rates.index, rates.values, color='#EF5350', edgecolor='white', alpha=0.85)
    ax.axhline(df['Attrition_flag'].mean() * 100, color='navy',
               linestyle='--', linewidth=1.2, label=f'Overall avg ({df["Attrition_flag"].mean()*100:.1f}%)')
    ax.set_ylabel('Attrition Rate (%)')
    ax.set_title(title or f'Attrition Rate by {col}')
    ax.legend()
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
    for bar, (cat, rate) in zip(bars, rates.items()):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f'{rate:.1f}%\n(n={counts[cat]})', ha='center', va='bottom', fontsize=8)
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
attrition_rate_by('Department')

In [ ]:
attrition_rate_by('JobRole', figsize=(12, 4))

In [ ]:
attrition_rate_by('BusinessTravel')

In [ ]:
attrition_rate_by('MaritalStatus')

In [ ]:
attrition_rate_by('OverTime')

In [ ]:
# Age vs Attrition — box plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=df, x='Attrition', y='Age', palette=['#2196F3', '#F44336'], ax=axes[0])
axes[0].set_title('Age Distribution by Attrition')

sns.boxplot(data=df, x='Attrition', y='MonthlyIncome', palette=['#2196F3', '#F44336'], ax=axes[1])
axes[1].set_title('Monthly Income Distribution by Attrition')

plt.tight_layout()
plt.show()

In [ ]:
# Years at Company vs Attrition
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=df, x='Attrition', y='YearsAtCompany', palette=['#2196F3', '#F44336'], ax=axes[0])
axes[0].set_title('Years at Company by Attrition')

sns.boxplot(data=df, x='Attrition', y='YearsSinceLastPromotion', palette=['#2196F3', '#F44336'], ax=axes[1])
axes[1].set_title('Years Since Last Promotion by Attrition')

plt.tight_layout()
plt.show()

In [ ]:
# Satisfaction scores vs Attrition
satisfaction_cols = ['JobSatisfaction', 'EnvironmentSatisfaction',
                     'WorkLifeBalance', 'RelationshipSatisfaction', 'JobInvolvement']

fig, axes = plt.subplots(1, len(satisfaction_cols), figsize=(16, 4))

for ax, col in zip(axes, satisfaction_cols):
    rates = df.groupby(col)['Attrition_flag'].mean() * 100
    ax.bar(rates.index, rates.values, color='#AB47BC', edgecolor='white', alpha=0.85)
    ax.axhline(df['Attrition_flag'].mean() * 100, color='navy',
               linestyle='--', linewidth=1)
    ax.set_title(col.replace('Satisfaction', '\nSatisfaction'), fontsize=9)
    ax.set_xlabel('Score (1=Low → 4=High)')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

axes[0].set_ylabel('Attrition Rate (%)')
plt.suptitle('Attrition Rate by Satisfaction & Involvement Scores', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Correlation Analysis

In [ ]:
# Correlation of numerical features with Attrition_flag
corr_with_attrition = (df[num_cols].corr()['Attrition_flag']
                       .drop('Attrition_flag')
                       .sort_values(key=abs, ascending=False))

fig, ax = plt.subplots(figsize=(8, 8))
colors = ['#EF5350' if v > 0 else '#42A5F5' for v in corr_with_attrition]
ax.barh(corr_with_attrition.index, corr_with_attrition.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with Attrition')
ax.set_title('Feature Correlation with Attrition\n(Red = positive, Blue = negative)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Full correlation heatmap for numerical features
top_corr_cols = corr_with_attrition.head(12).index.tolist() + ['Attrition_flag']
corr_matrix = df[top_corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Correlation Heatmap — Top Features vs Attrition', fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Chi-Square Test — Categorical Features vs Attrition

In [ ]:
results = []
for col in cat_cols:
    if col == 'Attrition':
        continue
    ct = pd.crosstab(df[col], df['Attrition'])
    chi2, p, dof, _ = chi2_contingency(ct)
    results.append({'Feature': col, 'Chi2': round(chi2, 2), 'p-value': round(p, 4), 'DOF': dof})

chi2_df = pd.DataFrame(results).sort_values('p-value')
chi2_df['Significant (p<0.05)'] = chi2_df['p-value'] < 0.05
chi2_df.style.apply(
    lambda row: ['background-color: #c8e6c9' if row['Significant (p<0.05)'] else '' for _ in row],
    axis=1
)

## 10. Deep Dive — Overtime & Monthly Income Interaction

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Attrition rate by OverTime × Department
pivot = df.groupby(['Department', 'OverTime'])['Attrition_flag'].mean().unstack() * 100
pivot.plot(kind='bar', ax=axes[0], color=['#42A5F5', '#EF5350'], edgecolor='white')
axes[0].set_title('Attrition Rate: Department × OverTime')
axes[0].set_ylabel('Attrition Rate (%)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(title='OverTime')

# Monthly income by Attrition × Department
sns.boxplot(data=df, x='Department', y='MonthlyIncome', hue='Attrition',
            palette={'No': '#42A5F5', 'Yes': '#EF5350'}, ax=axes[1])
axes[1].set_title('Monthly Income: Department × Attrition')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# Age vs Monthly Income scatter coloured by Attrition
fig, ax = plt.subplots(figsize=(9, 5))
colors = df['Attrition'].map({'Yes': '#EF5350', 'No': '#90CAF9'})
ax.scatter(df['Age'], df['MonthlyIncome'], c=colors, alpha=0.4, s=20, edgecolors='none')

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#EF5350', markersize=8, label='Attrition: Yes'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#90CAF9', markersize=8, label='Attrition: No')
]
ax.legend(handles=legend_elements)
ax.set_xlabel('Age')
ax.set_ylabel('Monthly Income')
ax.set_title('Age vs Monthly Income — coloured by Attrition', fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Key Insights Summary

In [ ]:
overall_rate = df['Attrition_flag'].mean() * 100
top_dept = (df.groupby('Department')['Attrition_flag'].mean() * 100).idxmax()
top_dept_rate = (df.groupby('Department')['Attrition_flag'].mean() * 100).max()
top_role = (df.groupby('JobRole')['Attrition_flag'].mean() * 100).idxmax()
top_role_rate = (df.groupby('JobRole')['Attrition_flag'].mean() * 100).max()
ot_yes = df[df['OverTime'] == 'Yes']['Attrition_flag'].mean() * 100
ot_no  = df[df['OverTime'] == 'No']['Attrition_flag'].mean() * 100
avg_age_left = df[df['Attrition'] == 'Yes']['Age'].mean()
avg_age_stay = df[df['Attrition'] == 'No']['Age'].mean()
avg_inc_left = df[df['Attrition'] == 'Yes']['MonthlyIncome'].mean()
avg_inc_stay = df[df['Attrition'] == 'No']['MonthlyIncome'].mean()

print('=' * 55)
print('        HR ANALYTICS — KEY EDA FINDINGS')
print('=' * 55)
print(f'  Overall attrition rate          : {overall_rate:.2f}%')
print(f'  Highest attrition department    : {top_dept} ({top_dept_rate:.1f}%)')
print(f'  Highest attrition job role      : {top_role} ({top_role_rate:.1f}%)')
print(f'  Attrition — OverTime Yes        : {ot_yes:.1f}%')
print(f'  Attrition — OverTime No         : {ot_no:.1f}%')
print(f'  Avg age who left / stayed       : {avg_age_left:.1f}  vs  {avg_age_stay:.1f}')
print(f'  Avg income who left / stayed    : {avg_inc_left:,.0f}  vs  {avg_inc_stay:,.0f}')
print('=' * 55)
print()
print('Narrative:')
print(f'  - Employees who do OverTime are {ot_yes/ot_no:.1f}x more likely to leave.')
print(f'  - Leavers earn ~{(avg_inc_stay - avg_inc_left)/avg_inc_stay*100:.0f}% less than retained employees on average.')
print(f'  - Younger employees (avg {avg_age_left:.0f}) leave more than older ones (avg {avg_age_stay:.0f}).')
print(f'  - {top_dept} has the highest raw attrition count; HR has the highest rate.')